# CS781 Phase 1 — Legal LM from scratch (JAX, Kaggle TPU v3-8)
Byte-level BPE (A1-style, rewritten for speed) → **masked discrete diffusion language model** (bidirectional Transformer, MDLM-style objective, confidence-based iterative unmasking) → time-budgeted final run → sweep decoding → `submission.zip`.

## 1. Setup and configuration

In [1]:
import os, sys, time, json, math, random, pickle, subprocess, zipfile, gc, heapq, importlib.util
from collections import Counter, defaultdict
T0 = time.time()                      # wall-clock start of the whole notebook (9h Kaggle limit)

def _pip(pkg): subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
for _m, _p in [("regex", "regex"), ("bert_score", "bert-score"), ("matplotlib", "matplotlib")]:
    if importlib.util.find_spec(_m) is None: _pip(_p)

import numpy as np
import regex as re2
from huggingface_hub import hf_hub_download, HfApi, login
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF"); login(token=HF_TOKEN)
except Exception as e:
    HF_TOKEN = None; print("HF login skipped:", e)

# ---------------- configuration ----------------
SEED = 42; random.seed(SEED); np.random.seed(SEED); rng = random.Random(SEED)
HF_REPO = "Exploration-Lab/CS-781-Capstone"
WORK = "/kaggle/working/cs781_phase1_diffusion"; TMP = "/tmp/hf"
os.makedirs(WORK, exist_ok=True); os.makedirs(TMP, exist_ok=True)

TOTAL_SEC   = 8.6 * 3600      # hard budget for the whole notebook (leaves ~24 min slack under 9h)
RESERVE_SEC = 60 * 60         # kept free after training for decoding sweep + test generation + scoring
FINAL_MAX_MIN = 330           # upper cap for the final training run (auto-shrunk by the budget guard)

# data mixing (share of the character budget per source) and sampling
MIX = {"acts": 0.05, "high_courts": 0.35, "supreme_court": 0.35, "tribunals": 0.25}
TOTAL_CHARS = 3.5e9           # ~ 800M tokens with a 16k byte-level BPE
VAL_P, VAL_CHARS_PER_SRC = 0.003, 4_000_000
TOK_CHARS = 80_000_000        # text sample used to train the tokenizer
VOCAB = 16384

# model / sequence
CTX, PROMPT_MAX, GEN_TOK = 1280, 960, 320      # prompt (512 words) is left-truncated to PROMPT_MAX tokens
D, L, H, HD, V = 768, 12, 12, 64, VOCAB
PCB = 8                                         # sequences per TPU core per step
PEAK_LR, MIN_LR_FRAC, WD, B1, B2, EPS, CLIP = 6e-4, 0.1, 0.1, 0.9, 0.95, 1e-8, 1.0
ABL_STEPS = 300                                 # steps per architecture in the ablation
TARGET_WORDS = 128; N_VAL_PROMPTS = 96
print("config ok")


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


config ok


## 2. Data: stream official Phase-1 files, sample by mixing policy, hold out validation docs

In [2]:
def iter_json_array(path, chunk=1 << 26):
    """Stream the objects of a top-level JSON array without loading the whole file."""
    dec = json.JSONDecoder()
    with open(path, "r", encoding="utf-8") as f:
        buf = f.read(chunk); i = 0
        while i < len(buf) and buf[i] in " \n\r\t": i += 1
        i += 1                                            # skip '['
        while True:
            while True:
                while i < len(buf) and buf[i] in " \n\r\t,": i += 1
                if i < len(buf): break
                more = f.read(chunk)
                if not more: return
                buf, i = more, 0
            if buf[i] == "]": return
            try:
                obj, j = dec.raw_decode(buf, i); i = j; yield obj
            except json.JSONDecodeError:
                more = f.read(chunk)
                if not more: return
                buf, i = buf[i:] + more, 0

TRAIN_NPY, VAL_NPY, VALDOC_JSON, TOK_PATH = f"{WORK}/train_tokens.npy", f"{WORK}/val_tokens.npy", f"{WORK}/val_docs.json", f"{WORK}/bpe_legal.json"
CACHED = all(os.path.exists(x) for x in [TRAIN_NPY, VAL_NPY, VALDOC_JSON, TOK_PATH])
api = HfApi()
files = api.list_repo_files(HF_REPO, repo_type="dataset", token=HF_TOKEN)
tr_files = {k: next(f for f in files if f.startswith("Phase_1_Pretraining/Train/") and k in f) for k in MIX}
test_file = next(f for f in files if f.startswith("Phase_1_Pretraining/Test/") and f.endswith(".json"))
print(tr_files, test_file)

train_docs, train_src, val_docs = [], [], []
stats = {}
for src, fn in ([] if CACHED else tr_files.items()):
    path = hf_hub_download(HF_REPO, fn, repo_type="dataset", token=HF_TOKEN, local_dir=TMP)
    size = os.path.getsize(path); budget = MIX[src] * TOTAL_CHARS
    p = min(1.0, 1.2 * budget / size)                     # uniform subsample across the file
    got = vch = seen = 0
    for rec in iter_json_array(path):
        seen += 1
        t = " ".join(rec["text"].split())                 # whitespace-normalised (test prompts are space-joined words)
        if len(t) < 200: continue
        u = rng.random()
        if u < VAL_P:
            if vch < VAL_CHARS_PER_SRC: val_docs.append((src, t)); vch += len(t)
        elif u < VAL_P + p:
            train_docs.append(t); train_src.append(src); got += len(t)
            if got >= budget: break
    stats[src] = dict(file_MB=size / 1e6, docs_scanned=seen, train_chars=got, val_chars=vch)
    print(src, stats[src]); os.remove(path); gc.collect()
if CACHED: val_docs = json.load(open(VALDOC_JSON, encoding="utf-8"))
else: json.dump(val_docs, open(VALDOC_JSON, "w", encoding="utf-8")); json.dump(stats, open(f"{WORK}/data_stats.json", "w"), indent=1)
print("train docs:", len(train_docs), "val docs:", len(val_docs))

{'acts': 'Phase_1_Pretraining/Train/train_acts.json', 'high_courts': 'Phase_1_Pretraining/Train/train_high_courts.json', 'supreme_court': 'Phase_1_Pretraining/Train/train_supreme_court.json', 'tribunals': 'Phase_1_Pretraining/Train/train_tribunals.json'} Phase_1_Pretraining/Test/test.json


Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

acts {'file_MB': 728.801891, 'docs_scanned': 16124, 'train_chars': 175101383, 'val_chars': 2837180}


Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

... (3 outputs left out)

supreme_court {'file_MB': 1322.567168, 'docs_scanned': 55027, 'train_chars': 1225015909, 'val_chars': 3120266}


Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

tribunals {'file_MB': 6483.962193, 'docs_scanned': 773801, 'train_chars': 875015523, 'val_chars': 4003830}
train docs: 501734 val docs: 2297


## 3. Tokenizer: byte-level BPE (A1 algorithm, heap-based incremental merges)

In [3]:
PAT = re2.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+""")

def bytes_to_unicode():
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256)); cs = bs[:]; n = 0
    for b in range(256):
        if b not in bs: bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, [chr(c) for c in cs]))
B2U = bytes_to_unicode(); U2B = {v: k for k, v in B2U.items()}

def merge_word(w, a, b, new):
    out, i, n = [], 0, len(w)
    while i < n:
        if i < n - 1 and w[i] == a and w[i + 1] == b: out.append(new); i += 2
        else: out.append(w[i]); i += 1
    return out

class ByteBPE:
    SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
    def __init__(self, vocab_size=16384): self.vocab_size = vocab_size; self.merges = []; self.itos = []; self.cache = {}
    def fit(self, word_counts, max_unique=300_000):
        items = word_counts.most_common(max_unique)
        words = [[B2U[b] for b in w.encode("utf-8")] for w, _ in items]; freqs = [f for _, f in items]
        self.itos = self.SPECIALS + [B2U[b] for b in range(256)]; seen = set(self.itos)
        pc = defaultdict(int); where = defaultdict(set)
        for i, w in enumerate(words):
            for q in zip(w, w[1:]): pc[q] += freqs[i]; where[q].add(i)
        heap = [(-c, q) for q, c in pc.items()]; heapq.heapify(heap)
        n_merges = self.vocab_size - len(self.itos)
        while len(self.merges) < n_merges and heap:
            negc, p = heapq.heappop(heap); c = pc.get(p, 0)
            if c <= 0: continue
            if c != -negc: heapq.heappush(heap, (-c, p)); continue
            a, b = p; new = a + b; self.merges.append([a, b])
            if new not in seen: seen.add(new); self.itos.append(new)
            touched = set()
            for i in list(where[p]):
                w = words[i]; nw = merge_word(w, a, b, new)
                if len(nw) == len(w): continue
                f = freqs[i]
                for q in zip(w, w[1:]): pc[q] -= f
                for q in zip(nw, nw[1:]): pc[q] += f; where[q].add(i); touched.add(q)
                words[i] = nw
            where.pop(p, None); pc.pop(p, None)
            for q in touched:
                if pc[q] > 0: heapq.heappush(heap, (-pc[q], q))
            if len(self.merges) % 2000 == 0: print("merges", len(self.merges), flush=True)
        self.finalize(); return self
    def finalize(self):
        self.stoi = {s: i for i, s in enumerate(self.itos)}
        self.rank = {tuple(m): i for i, m in enumerate(self.merges)}
        self.tbl = [b"" if i < 4 else bytes(U2B[ch] for ch in s) for i, s in enumerate(self.itos)]
    def save(self, path):
        json.dump({"merges": self.merges, "itos": self.itos, "vocab_size": len(self.itos)}, open(path, "w", encoding="utf-8"), ensure_ascii=False)
    @classmethod
    def load(cls, path):
        d = json.load(open(path, encoding="utf-8")); t = cls(d["vocab_size"]); t.merges = d["merges"]; t.itos = d["itos"]; t.finalize(); return t
    def word_ids(self, w):
        r = self.cache.get(w)
        if r is None:
            s = [B2U[b] for b in w.encode("utf-8")]
            while len(s) > 1:
                best, br = None, 1 << 60
                for q in zip(s, s[1:]):
                    rk = self.rank.get(q)
                    if rk is not None and rk < br: br, best = rk, q
                if best is None: break
                s = merge_word(s, best[0], best[1], best[0] + best[1])
            r = [self.stoi[x] for x in s]; self.cache[w] = r
        return r
    def encode(self, text):
        out = []
        for w in PAT.findall(text): out.extend(self.word_ids(w))
        return out
    def decode(self, ids): return b"".join(self.tbl[i] for i in ids if i < len(self.tbl)).decode("utf-8", "replace")

if os.path.exists(TOK_PATH):
    TOK = ByteBPE.load(TOK_PATH)
else:
    t0 = time.time(); idx = list(range(len(train_docs))); rng.shuffle(idx); wc = Counter(); ch = 0
    for i in idx:
        wc.update(PAT.findall(train_docs[i])); ch += len(train_docs[i])
        if ch >= TOK_CHARS: break
    print("unique pretokens:", len(wc), f"({time.time()-t0:.0f}s)")
    TOK = ByteBPE(VOCAB).fit(wc); TOK.save(TOK_PATH); del wc
print("vocab:", len(TOK.itos), "merges:", len(TOK.merges)); EOS = 3
print(TOK.decode(TOK.encode("The Hon'ble Supreme Court held under Section 138 of the N.I. Act, 1881 that ₹5,00,000 is payable.")))

unique pretokens: 217329 (7s)


merges 2000


merges 4000


merges 6000


merges 8000


merges 10000


merges 12000


merges 14000


merges 16000


vocab: 16384 merges: 16124
The Hon'ble Supreme Court held under Section 138 of the N.I. Act, 1881 that ₹5,00,000 is payable.


## 4. Encode corpus (multiprocessing, before JAX is initialised)

In [4]:
import multiprocessing as mp
DOCS = train_docs
def _enc_range(a):
    s, e = a; outs, lens = [], []
    for t in DOCS[s:e]:
        ids = TOK.encode(t); ids.append(EOS); outs.append(np.asarray(ids, dtype=np.uint16)); lens.append(len(ids))
    return np.concatenate(outs), np.asarray(lens)

if os.path.exists(TRAIN_NPY) and os.path.exists(VAL_NPY):
    train_tokens = np.load(TRAIN_NPY); val_tokens = np.load(VAL_NPY)
else:
    t0 = time.time(); CH = 400
    ranges = [(s, min(s + CH, len(DOCS))) for s in range(0, len(DOCS), CH)]
    with mp.get_context("fork").Pool(min(os.cpu_count() or 1, 32)) as pool:
        res = list(pool.imap(_enc_range, ranges))
    train_tokens = np.concatenate([r[0] for r in res]); doc_lens = np.concatenate([r[1] for r in res])
    per_src = defaultdict(int)
    for s, n in zip(train_src, doc_lens): per_src[s] += int(n)
    print("tokens per source:", dict(per_src), f"| total {len(train_tokens)/1e6:.1f}M | chars/token {sum(len(d) for d in DOCS)/len(train_tokens):.2f} | {time.time()-t0:.0f}s")
    val_tokens = np.concatenate([np.asarray(TOK.encode(t) + [EOS], dtype=np.uint16) for _, t in val_docs])
    np.save(TRAIN_NPY, train_tokens); np.save(VAL_NPY, val_tokens)
print("train tokens:", len(train_tokens), "val tokens:", len(val_tokens))
DOCS = None; train_docs = train_src = None; gc.collect()

# validation prompts mimicking the test format: 512-word prompt -> next 128 words
vr = random.Random(SEED + 1); cand = [t.split() for _, t in val_docs]; cand = [w for w in cand if len(w) >= 512 + TARGET_WORDS]
vr.shuffle(cand); val_prompts, val_refs = [], []
for w in cand[:N_VAL_PROMPTS]:
    s = vr.randint(0, len(w) - 512 - TARGET_WORDS)
    val_prompts.append(" ".join(w[s:s + 512])); val_refs.append(" ".join(w[s + 512:s + 512 + TARGET_WORDS]))
print("val prompts:", len(val_prompts))

tokens per source: {'acts': 38814725, 'high_courts': 286133362, 'supreme_court': 264694033, 'tribunals': 222234798} | total 811.9M | chars/token 4.31 | 20s


train tokens: 811876918 val tokens: 3196452


val prompts: 96


## 5. Model: masked diffusion LM — bidirectional Transformer with optional adaLN noise-level conditioning

In [5]:
import jax, jax.numpy as jnp
from jax import lax
from functools import partial
NDEV = jax.local_device_count(); GB = NDEV * PCB; BF = jnp.bfloat16; F32 = jnp.float32
print("devices:", jax.devices()[:2], "x", NDEV)
MASK = V                      # extra embedding row (id 16384) used as the [MASK] token; never predicted
CD = 128                      # width of the noise-level (mask-rate) conditioning vector
LOCAL_WINDOW = 256            # bidirectional local window (|i-j| < 256) for the hybrid variant

# All variants: bidirectional attention (no causal mask), GQA(12 q-heads / 4 kv-heads), RoPE, QK-norm, RMSNorm, SwiGLU.
#   mdlm_plain             : LLaDA-style, no noise-level input
#   ours_mdlm_adaln        : + adaLN-Zero style modulation (shift/scale/gate per block) from the sinusoidal embedding of the mask rate t
#   ours_mdlm_adaln_hybrid : + alternating local(window 256)/global bidirectional attention (2 local : 1 global)
VARIANTS = {
  "mdlm_plain":             dict(adaln=False, hybrid=False, kvh=4, ff=2048),
  "ours_mdlm_adaln":        dict(adaln=True,  hybrid=False, kvh=4, ff=2048),
  "ours_mdlm_adaln_hybrid": dict(adaln=True,  hybrid=True,  kvh=4, ff=2048),
}
MATS = {"wq", "wk", "wv", "wo", "w1", "w2", "w3", "wada", "wadaf", "wt1", "wt2"}

def make_windows(c): return np.array([LOCAL_WINDOW if (c["hybrid"] and i % 3 != 2) else 1 << 30 for i in range(L)], np.int32)

def init_params(key, c):
    ks = jax.random.split(key, 16); KV, ff = c["kvh"], c["ff"]
    nrm = lambda k, s, sd=0.02: jax.random.normal(k, s, F32) * sd
    so = 0.02 / math.sqrt(2 * L)
    lp = {"wq": nrm(ks[0], (L, D, H * HD)), "wk": nrm(ks[1], (L, D, KV * HD)), "wv": nrm(ks[2], (L, D, KV * HD)), "wo": nrm(ks[3], (L, H * HD, D), so),
          "n1": jnp.ones((L, D)), "n2": jnp.ones((L, D)), "qn": jnp.ones((L, HD)), "kn": jnp.ones((L, HD)),
          "w1": nrm(ks[4], (L, D, ff)), "w3": nrm(ks[5], (L, D, ff)), "w2": nrm(ks[6], (L, ff, D), so)}
    if c["adaln"]: lp["wada"] = jnp.zeros((L, CD, 6 * D))      # zero-init: network starts identical to the unconditioned one
    p = {"emb": nrm(ks[7], (V + 1, D)), "layers": lp, "nf": jnp.ones((D,))}
    if c["adaln"]:
        p.update(wt1=nrm(ks[8], (CD, CD), 0.1), bt1=jnp.zeros((CD,)), wt2=nrm(ks[9], (CD, CD), 0.1), bt2=jnp.zeros((CD,)), wadaf=jnp.zeros((CD, 2 * D)))
    return p

def n_params(p): return int(sum(np.prod(x.shape) for x in jax.tree_util.tree_leaves(p)))

_inv = 1.0 / (10000.0 ** (np.arange(0, HD, 2) / HD)); _t = np.arange(CTX)[:, None] * _inv[None]
ROPE_COS, ROPE_SIN = jnp.asarray(np.cos(_t), F32), jnp.asarray(np.sin(_t), F32)

def apply_rope(x, cos, sin):
    x1, x2 = x[..., :HD // 2].astype(F32), x[..., HD // 2:].astype(F32); cos, sin = cos[:, :, None, :], sin[:, :, None, :]
    return jnp.concatenate([x1 * cos - x2 * sin, x2 * cos + x1 * sin], -1).astype(BF)
def rms_n(x, w): x = x.astype(F32); return (x * lax.rsqrt((x ** 2).mean(-1, keepdims=True) + 1e-6) * w).astype(BF)
rms_h = rms_n

def attend(q, K, Vv, m):
    B, T = q.shape[:2]
    s = jnp.einsum("btgrd,bsgd->bgrts", q, K, preferred_element_type=F32) * (HD ** -0.5)
    a = jax.nn.softmax(jnp.where(m[:, None, None], s, -1e30), -1).astype(BF)
    return jnp.einsum("bgrts,bsgd->btgrd", a, Vv).reshape(B, T, -1)

def sinus(t):
    f = jnp.exp(-math.log(10000.0) * jnp.arange(CD // 2) / (CD // 2)); a = t[:, None] * 1000.0 * f[None]
    return jnp.concatenate([jnp.cos(a), jnp.sin(a)], -1)

def dlayer(c, x, lp, win, cos, sin, pos, kval, cond):
    B, T, _ = x.shape; KV = c["kvh"]; R = H // KV
    if cond is None: sh1 = sc1 = sh2 = sc2 = 0.0; g1 = g2 = 1.0
    else:
        a = jax.nn.silu(cond) @ lp["wada"]; sh1, sc1, g1, sh2, sc2, g2 = [z[:, None, :] for z in jnp.split(a, 6, -1)]; g1, g2 = 1.0 + g1, 1.0 + g2
    h = rms_n(x, lp["n1"])
    if cond is not None: h = (h.astype(F32) * (1.0 + sc1) + sh1).astype(BF)
    q = rms_h((h @ lp["wq"].astype(BF)).reshape(B, T, H, HD), lp["qn"]); k = rms_h((h @ lp["wk"].astype(BF)).reshape(B, T, KV, HD), lp["kn"])
    v = (h @ lp["wv"].astype(BF)).reshape(B, T, KV, HD); q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
    m = (jnp.abs(pos[:, None, :] - pos[:, :, None]) < win) & kval[:, None, :]          # bidirectional (optionally windowed) mask
    o = attend(q.reshape(B, T, KV, R, HD), k, v, m) @ lp["wo"].astype(BF)
    x = x + g1 * o.astype(F32)
    h = rms_n(x, lp["n2"])
    if cond is not None: h = (h.astype(F32) * (1.0 + sc2) + sh2).astype(BF)
    mo = (jax.nn.silu(h @ lp["w1"].astype(BF)) * (h @ lp["w3"].astype(BF))) @ lp["w2"].astype(BF)
    return x + g2 * mo.astype(F32)

def dm_forward(c, win, params, tokens, t, pos, kval, hidden_from=0):
    """tokens (B,T) with MASK ids; t (B,) current mask rate; returns vocabulary logits for positions >= hidden_from."""
    x = params["emb"][tokens]; cos, sin = ROPE_COS[pos], ROPE_SIN[pos]; cond = None
    if c["adaln"]: cond = jax.nn.silu(sinus(t) @ params["wt1"] + params["bt1"]) @ params["wt2"] + params["bt2"]
    def body(x, xs): lp, w = xs; return dlayer(c, x, lp, w, cos, sin, pos, kval, cond), None
    x, _ = lax.scan(jax.checkpoint(body, prevent_cse=False), x, (params["layers"], jnp.asarray(win)))
    h = rms_n(x[:, hidden_from:], params["nf"])
    if cond is not None:
        sh, sc = jnp.split(jax.nn.silu(cond) @ params["wadaf"], 2, -1); h = (h.astype(F32) * (1.0 + sc[:, None]) + sh[:, None]).astype(BF)
    return h @ params["emb"][:V].astype(BF).T

def dm_loss(c, win, params, b, key, train):
    """MDLM objective: mask each eligible token w.p. t~U(0,1] (stratified over the batch), loss = (1/t) * CE on masked tokens, per-token normalised.
    train: 50% of sequences mask every token (unconditional LM), 50% keep the 960-token prompt clean and mask only the 320-token continuation.
    eval : continuation-only masking -> negative-ELBO (nats/token, an upper bound on the continuation NLL)."""
    x0 = b[:, :CTX]; B = x0.shape[0]; k1, k2, k3 = jax.random.split(key, 3)
    t = jnp.clip((jax.random.uniform(k1, ()) + jnp.arange(B) / B) % 1.0, 0.02, 1.0)
    pos = jnp.arange(CTX); suffix = pos >= PROMPT_MAX
    condm = jax.random.bernoulli(k2, 0.5, (B,)) if train else jnp.ones((B,), bool)
    elig = jnp.where(condm[:, None], suffix[None], True)
    masked = (jax.random.uniform(k3, (B, CTX)) < t[:, None]) & elig
    hf = 0 if train else PROMPT_MAX
    logits = dm_forward(c, win, params, jnp.where(masked, MASK, x0), t, pos[None], jnp.ones((1, CTX), bool), hf).astype(F32)
    ce = jax.nn.logsumexp(logits, -1) - jnp.take_along_axis(logits, x0[:, hf:, None], -1)[..., 0]
    return jnp.mean((ce * masked[:, hf:]).sum(-1) / t / elig.sum(-1).astype(F32))

for _n, _c in VARIANTS.items():
    _p = init_params(jax.random.PRNGKey(0), _c); _k = n_params(_p); assert 50e6 <= _k <= 300e6, _k
    print(f"{_n:24s} {_k/1e6:6.1f}M params (within 50M-300M)")
del _p

/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


E0000 00:00:1791236562.832664      73 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0)] x 8


mdlm_plain                 88.1M params (within 50M-300M)


ours_mdlm_adaln            95.4M params (within 50M-300M)
ours_mdlm_adaln_hybrid     95.4M params (within 50M-300M)


## 6. Training utilities (pmap data-parallel, hand-written AdamW, cosine schedule, time-budget aware)

In [6]:
class Batcher:
    def __init__(s, toks, seed): s.t = toks; s.n = (len(toks) - 1) // CTX; s.r = np.random.RandomState(seed); s.perm = s.r.permutation(s.n); s.ptr = 0
    def next(s):
        if s.ptr + GB > s.n: s.perm = s.r.permutation(s.n); s.ptr = 0
        i = s.perm[s.ptr:s.ptr + GB]; s.ptr += GB
        return s.t[i[:, None] * CTX + np.arange(CTX + 1)[None]].astype(np.int32).reshape(NDEV, PCB, CTX + 1)

_nb = (len(val_tokens) - 1) // CTX
VAL = val_tokens[(np.arange(_nb)[:, None] * CTX + np.arange(CTX + 1)[None])].astype(np.int32)
VAL_BATCHES = min(len(VAL) // GB, 24); assert VAL_BATCHES > 0
VAL = VAL[:VAL_BATCHES * GB].reshape(VAL_BATCHES, NDEV, PCB, CTX + 1)
BASE_KEY = jax.random.PRNGKey(SEED)

def rep(tree):
    try: return jax.device_put_replicated(tree, jax.local_devices())
    except Exception:
        return jax.pmap(lambda t: t)(jax.tree_util.tree_map(lambda x: np.broadcast_to(np.asarray(x), (NDEV,) + np.shape(x)), tree))
def unrep(tree): return jax.tree_util.tree_map(lambda x: np.asarray(x[0]), tree)

def make_fns(c, win):
    tmap = jax.tree_util.tree_map
    def lr_at(step, total, warm):
        w = step / warm; pr = jnp.clip((step - warm) / jnp.maximum(total - warm, 1.0), 0.0, 1.0)
        return PEAK_LR * jnp.where(step < warm, w, MIN_LR_FRAC + (1 - MIN_LR_FRAC) * 0.5 * (1 + jnp.cos(jnp.pi * pr)))
    @partial(jax.pmap, axis_name="d", in_axes=(0, 0, 0, 0, None, None, None), donate_argnums=(0, 1, 2))
    def train_step(params, m, v, batch, step, total, warm):
        key = jax.random.fold_in(jax.random.fold_in(BASE_KEY, step.astype(jnp.int32)), lax.axis_index("d"))
        loss, g = jax.value_and_grad(lambda p: dm_loss(c, win, p, batch, key, True))(params)
        g = lax.pmean(g, "d"); loss = lax.pmean(loss, "d")
        gn = jnp.sqrt(sum(jnp.sum(x.astype(F32) ** 2) for x in jax.tree_util.tree_leaves(g)))
        sc = jnp.minimum(1.0, CLIP / (gn + 1e-6)); lr = lr_at(step, total, warm); t = step + 1.0
        nm = tmap(lambda m_, g_: B1 * m_ + (1 - B1) * g_ * sc, m, g)
        nv = tmap(lambda v_, g_: B2 * v_ + (1 - B2) * (g_ * sc) ** 2, v, g)
        npar = tmap(lambda p, m_, v_, dm: p - lr * ((m_ / (1 - B1 ** t)) / (jnp.sqrt(v_ / (1 - B2 ** t)) + EPS) + WD * dm * p), params, nm, nv, DECAY)
        return npar, nm, nv, loss, gn
    @partial(jax.pmap, axis_name="d")
    def eval_step(params, batch):
        key = jax.random.fold_in(jax.random.PRNGKey(1234), lax.axis_index("d"))
        return lax.pmean(dm_loss(c, win, params, batch, key, False), "d")
    return train_step, eval_step

def evaluate(eval_step, params):
    return float(np.mean([float(eval_step(params, VAL[i])[0]) for i in range(VAL_BATCHES)]))

def run_training(name, c, fixed_steps=None, budget_sec=None, warmup=200, ckpt=None, eval_every=500, log_every=50, ckpt_every_sec=1500):
    global DECAY
    win = make_windows(c)
    p0 = init_params(jax.random.PRNGKey(SEED), c); n = n_params(p0); assert 50e6 <= n <= 300e6
    DECAY = jax.tree_util.tree_map_with_path(lambda path, x: 1.0 if getattr(path[-1], "key", None) in MATS else 0.0, p0)
    train_step, eval_step = make_fns(c, win)
    step, total, hist = 0, (fixed_steps or 10 ** 9), []
    if ckpt and os.path.exists(ckpt):
        d = pickle.load(open(ckpt, "rb")); params, m, v = rep(d["params"]), rep(d["m"]), rep(d["v"]); step, total, hist = d["step"], d["total"], d["hist"]
        print(f"resumed {name} at step {step}/{total}")
    else:
        params = rep(p0); m = rep(jax.tree_util.tree_map(jnp.zeros_like, p0)); v = rep(jax.tree_util.tree_map(jnp.zeros_like, p0))
    del p0
    bt = Batcher(train_tokens, SEED + step); run_t0 = time.time(); last_ck = time.time(); tb = None
    need_bench = budget_sec is not None and total == 10 ** 9
    while step < total:
        params, m, v, loss, gn = train_step(params, m, v, bt.next(), jnp.float32(step), jnp.float32(total), jnp.float32(warmup))
        step += 1
        if need_bench and step == 6: loss.block_until_ready(); tb = time.time()
        if need_bench and step == 26:
            loss.block_until_ready(); rate = (time.time() - tb) / 20.0
            total = step + int(0.96 * (budget_sec - (time.time() - run_t0)) / rate); need_bench = False
            print(f"[{name}] {rate:.3f}s/step, {GB*CTX/rate:,.0f} tok/s -> total_steps={total} (~{total*GB*CTX/1e6:.0f}M tokens)")
        if step % log_every == 0:
            lv = float(loss[0]); hist.append(dict(step=step, tokens=step * GB * CTX, train_loss=lv, val_loss=None, time=time.time() - run_t0))
            if step % (log_every * 4) == 0: print(f"[{name}] step {step}/{total} loss {lv:.4f} gn {float(gn[0]):.2f} | {(time.time()-run_t0)/60:.1f} min", flush=True)
        if step % eval_every == 0 or step == total:
            vl = evaluate(eval_step, params); hist.append(dict(step=step, tokens=step * GB * CTX, train_loss=None, val_loss=vl, time=time.time() - run_t0))
            print(f"[{name}] step {step} VAL negELBO {vl:.4f} nats/token (ppl bound {math.exp(vl):.1f})", flush=True)
        if ckpt and time.time() - last_ck > ckpt_every_sec:
            pickle.dump(dict(params=unrep(params), m=unrep(m), v=unrep(v), step=step, total=total, hist=hist), open(ckpt + ".tmp", "wb")); os.replace(ckpt + ".tmp", ckpt); last_ck = time.time()
        if time.time() - T0 > TOTAL_SEC - RESERVE_SEC: print("time guard hit, stopping early"); break
    vl = evaluate(eval_step, params); secs = time.time() - run_t0
    return params, dict(name=name, params_M=n / 1e6, steps=step, tokens=step * GB * CTX, val_loss=vl, val_ppl=math.exp(vl), seconds=secs, tok_per_s=step * GB * CTX / secs, hist=hist)
print("utilities ready")

utilities ready


## 7. Architecture ablation (equal steps per variant, same data order, seed and masking noise)

In [7]:
ABL_PATH = f"{WORK}/ablation_results.json"
if os.path.exists(ABL_PATH): ABL = json.load(open(ABL_PATH))
else:
    ABL = {}
    for name, c in VARIANTS.items():
        p, info = run_training("abl_" + name, c, fixed_steps=ABL_STEPS, warmup=30, eval_every=100, log_every=25)
        info.pop("hist"); ABL[name] = info; del p; gc.collect()
        print(f">>> {name}: val negELBO {info['val_loss']:.4f} | {info['tok_per_s']:,.0f} tok/s | {info['params_M']:.1f}M")
        json.dump(ABL, open(ABL_PATH, "w"), indent=1)
print("val negELBO = continuation-only masked-diffusion bound, nats/token (upper bound on continuation NLL)")
print(f"{'variant':26s}{'paramsM':>9s}{'negELBO':>10s}{'ppl<=':>9s}{'tok/s':>10s}")
for k, r in ABL.items(): print(f"{k:26s}{r['params_M']:9.1f}{r['val_loss']:10.4f}{r['val_ppl']:9.2f}{r['tok_per_s']:10,.0f}")
BEST = min([k for k in ABL if k.startswith("ours_")], key=lambda k: ABL[k]["val_loss"])
print("selected for final run:", BEST)

[abl_mdlm_plain] step 100/300 loss 6.6461 gn 0.69 | 0.6 min


[abl_mdlm_plain] step 100 VAL negELBO 6.5144 nats/token (ppl bound 674.8)


[abl_mdlm_plain] step 200/300 loss 6.3125 gn 0.76 | 1.0 min


[abl_mdlm_plain] step 200 VAL negELBO 6.0821 nats/token (ppl bound 437.9)


[abl_mdlm_plain] step 300/300 loss 6.1144 gn 0.79 | 1.3 min


[abl_mdlm_plain] step 300 VAL negELBO 5.8569 nats/token (ppl bound 349.6)


... (11 outputs left out)

[abl_ours_mdlm_adaln_hybrid] step 200 VAL negELBO 6.0034 nats/token (ppl bound 404.8)


[abl_ours_mdlm_adaln_hybrid] step 300/300 loss 5.9941 gn 1.37 | 1.4 min


[abl_ours_mdlm_adaln_hybrid] step 300 VAL negELBO 5.7535 nats/token (ppl bound 315.3)


>>> ours_mdlm_adaln_hybrid: val negELBO 5.7535 | 281,647 tok/s | 95.4M
val negELBO = continuation-only masked-diffusion bound, nats/token (upper bound on continuation NLL)
variant                     paramsM   negELBO    ppl<=     tok/s
mdlm_plain                     88.1    5.8569   349.64   319,062
ours_mdlm_adaln                95.4    5.9537   385.17   296,076
ours_mdlm_adaln_hybrid         95.4    5.7535   315.31   281,647
selected for final run: ours_mdlm_adaln_hybrid


## 8. Final training (time-budgeted, checkpointed, resumable)

In [8]:
elapsed = time.time() - T0
budget = min(FINAL_MAX_MIN * 60, TOTAL_SEC - elapsed - RESERVE_SEC)
print(f"elapsed {elapsed/60:.0f} min -> final training budget {budget/60:.0f} min")
FINAL_CKPT = f"{WORK}/final_ckpt.pkl"
C = VARIANTS[BEST]; WIN = make_windows(C)
params_rep, FINAL = run_training("final_" + BEST, C, budget_sec=budget, warmup=300, ckpt=FINAL_CKPT, eval_every=500, log_every=50)
pickle.dump(dict(params=unrep(params_rep), cfg=C, variant=BEST), open(f"{WORK}/final_params.pkl", "wb"))
json.dump({k: v for k, v in FINAL.items() if k != "hist"}, open(f"{WORK}/final_summary.json", "w"), indent=1)
print({k: v for k, v in FINAL.items() if k != "hist"})

import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
h = FINAL["hist"]; tr = [(x["tokens"] / 1e6, x["train_loss"]) for x in h if x["train_loss"] is not None]; va = [(x["tokens"] / 1e6, x["val_loss"]) for x in h if x["val_loss"] is not None]
plt.figure(figsize=(9, 5)); plt.plot(*zip(*tr), label="train loss (mixed objective)", alpha=.7); plt.plot(*zip(*va), "o-", label="val negELBO (continuation)")
plt.xlabel("tokens (M)"); plt.ylabel("nats / token"); plt.grid(True); plt.legend(); plt.title(f"Phase 1 — {BEST}"); plt.savefig(f"{WORK}/loss_curve.png", dpi=200, bbox_inches="tight"); plt.show()

elapsed 17 min -> final training budget 330 min


[final_ours_mdlm_adaln_hybrid] 0.179s/step, 458,209 tok/s -> total_steps=106178 (~8698M tokens)


[final_ours_mdlm_adaln_hybrid] step 200/106178 loss 6.0182 gn 1.58 | 1.0 min


[final_ours_mdlm_adaln_hybrid] step 400/106178 loss 4.7314 gn 1.57 | 1.6 min


[final_ours_mdlm_adaln_hybrid] step 500 VAL negELBO 4.5421 nats/token (ppl bound 93.9)


[final_ours_mdlm_adaln_hybrid] step 600/106178 loss 4.4628 gn 1.16 | 2.3 min


... (736 outputs left out)

[final_ours_mdlm_adaln_hybrid] step 106000/106178 loss 2.1092 gn 1.80 | 319.7 min


[final_ours_mdlm_adaln_hybrid] step 106000 VAL negELBO 2.0892 nats/token (ppl bound 8.1)


[final_ours_mdlm_adaln_hybrid] step 106178 VAL negELBO 2.0882 nats/token (ppl bound 8.1)


{'name': 'final_ours_mdlm_adaln_hybrid', 'params_M': 95.409408, 'steps': 106178, 'tokens': 8698101760, 'val_loss': 2.08817565937837, 'val_ppl': 8.070178971153378, 'seconds': 19214.119676589966, 'tok_per_s': 452693.2228176742}


## 9. Generation: iterative confidence-based unmasking (optionally semi-autoregressive in blocks); decoding sweep on held-out prompts

In [9]:
def make_gen(c, win, spb, nb):
    """Start from [prompt | 320 x MASK]. nb sequential blocks, spb denoising steps per block. Each step predicts every masked token,
    then commits the most confident ones in the current block (low-confidence remasking); the rest stay masked."""
    BK = GEN_TOK // nb; assert BK * nb == GEN_TOK; winj = jnp.asarray(win)
    @partial(jax.pmap, in_axes=(0, 0, 0, None, None, 0))
    def gen(params, prompt, pad, temp, rp, key):
        b = prompt.shape[0]; slots = jnp.arange(CTX); kval = slots[None] >= pad[:, None]; pos = jnp.maximum(slots[None] - pad[:, None], 0)
        gpos = jnp.arange(GEN_TOK); rows = jnp.arange(b)[:, None]
        def step(carry, s):
            cont, masked, key = carry; key, sk = jax.random.split(key)
            blk, within = s // spb, s % spb; inblk = (gpos >= blk * BK) & (gpos < (blk + 1) * BK)
            t = masked.sum(-1).astype(F32) / GEN_TOK
            lg = dm_forward(c, winj, params, jnp.concatenate([prompt, cont], 1), t, pos, kval, PROMPT_MAX).astype(F32)
            lg = lg.at[:, :, :4].set(-1e30)
            seen = jnp.zeros((b, V), bool).at[rows, jnp.where(masked, 0, cont)].set(True)
            lg = jnp.where(seen[:, None, :], jnp.where(lg > 0, lg / rp, lg * rp), lg)       # repetition penalty on committed tokens
            tmp = jnp.maximum(temp, 1e-4); probs = jax.nn.softmax(lg / tmp, -1)
            pred = jnp.where(temp <= 0, jnp.argmax(lg, -1), jax.random.categorical(sk, lg / tmp, -1))
            conf = jnp.take_along_axis(probs, pred[..., None], -1)[..., 0]
            cand = masked & inblk[None]
            k = jnp.maximum(((within + 1) * BK) // spb - (BK - cand.sum(-1)), 0)
            rank = jnp.argsort(jnp.argsort(-jnp.where(cand, conf, -1.0), -1), -1)
            sel = cand & (rank < k[:, None])
            return (jnp.where(sel, pred, cont), masked & ~sel, key), None
        init = (jnp.full((b, GEN_TOK), MASK, jnp.int32), jnp.ones((b, GEN_TOK), bool), key)
        (cont, _, _), _ = lax.scan(step, init, jnp.arange(nb * spb))
        return cont
    return gen

GEN_FNS = {}
def generate(prompts, spb, nb, temp=0.0, rp=1.0, bs=8, seed=0):
    if (spb, nb) not in GEN_FNS: GEN_FNS[(spb, nb)] = make_gen(C, WIN, spb, nb)
    gen = GEN_FNS[(spb, nb)]
    ids_all = [TOK.encode(p)[-PROMPT_MAX:] for p in prompts]; per = NDEV * bs; out = []
    for s in range(0, len(ids_all), per):
        chunk = ids_all[s:s + per]; real = len(chunk); chunk = chunk + [chunk[-1]] * (per - real)
        arr = np.zeros((per, PROMPT_MAX), np.int32); pad = np.zeros(per, np.int32)
        for i, ids in enumerate(chunk): pl = PROMPT_MAX - len(ids); arr[i, pl:] = ids; pad[i] = pl
        keys = jax.random.split(jax.random.PRNGKey(seed + s), NDEV)
        res = np.asarray(gen(params_rep, arr.reshape(NDEV, bs, -1), pad.reshape(NDEV, bs), jnp.float32(temp), jnp.float32(rp), keys)).reshape(per, -1)[:real]
        for r in res:
            w = TOK.decode([int(t) for t in r if t < V]).split()[:TARGET_WORDS]; out.append(" ".join(w) if w else "the")
    return out

# ---- local metrics (same definitions as evaluate_phase1.py) ---- (same definitions as evaluate_phase1.py) ----
def lcs(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b): cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]
def rouge_l(p, r):
    p, r = p.split(), r.split(); l = lcs(p, r)
    return 0.0 if l == 0 else 2 * (l / len(p)) * (l / len(r)) / ((l / len(p)) + (l / len(r)))
def bleu4(preds, refs):
    from collections import Counter as C_; cl, tot, pl, rl = [0] * 4, [0] * 4, 0, 0
    for p, r in zip(preds, refs):
        p, r = p.split(), r.split(); pl += len(p); rl += len(r)
        for n in range(1, 5):
            pn = C_(tuple(p[i:i + n]) for i in range(len(p) - n + 1)); rn = C_(tuple(r[i:i + n]) for i in range(len(r) - n + 1)); tot[n - 1] += sum(pn.values())
            cl[n - 1] += sum(min(c, rn.get(g, 0)) for g, c in pn.items())
    pr = [cl[i] / tot[i] if tot[i] else 0 for i in range(4)]
    return 0.0 if min(pr) <= 0 or pl == 0 else (1.0 if pl > rl else math.exp(1 - rl / pl)) * math.exp(sum(math.log(x) for x in pr) / 4)
SCORER = None
def score(preds, refs):
    global SCORER
    out = dict(rougeL=float(np.mean([rouge_l(p, r) for p, r in zip(preds, refs)])), bleu4=bleu4(preds, refs), bertscore=None)
    try:
        if SCORER is None:
            import bert_score; SCORER = bert_score.BERTScorer(lang="en", device="cpu", batch_size=8)
        out["bertscore"] = float(SCORER.score(preds, refs)[2].mean())
    except Exception as e: print("BERTScore unavailable:", e)
    return out


SWEEP = [(48, 1, 0.0, 1.0), (12, 4, 0.0, 1.0), (12, 4, 0.0, 1.15), (8, 8, 0.0, 1.15), (8, 8, 0.3, 1.15)]   # (steps/block, blocks, temperature, repetition penalty)
SWEEP_RES = []
for cfg in SWEEP:
    preds = generate(val_prompts, *cfg); sc = score(preds, val_refs); SWEEP_RES.append((cfg, sc))
    print(f"steps/block={cfg[0]} blocks={cfg[1]} temp={cfg[2]} rep={cfg[3]} ->", {k: (round(v, 4) if v is not None else None) for k, v in sc.items()}, flush=True)
key_fn = lambda r: r[1]["bertscore"] if r[1]["bertscore"] is not None else r[1]["rougeL"]
BEST_DEC = max(SWEEP_RES, key=key_fn)[0]; print("best decoding:", BEST_DEC)
json.dump([[list(k), v] for k, v in SWEEP_RES], open(f"{WORK}/decoding_sweep.json", "w"), indent=1)
print("\nSAMPLE\nPROMPT TAIL:", " ".join(val_prompts[0].split()[-40:]), "\nGENERATED:", generate(val_prompts[:NDEV * 8], *BEST_DEC)[0], "\nREFERENCE:", val_refs[0])

/usr/local/lib/python3.12/site-packages/torch_xla/experimental/gru.py:113: SyntaxWarning: invalid escape sequence '\_'
  * **h_n**: tensor of shape :math:`(D * \text{num\_layers}, H_{out})` or


/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

... (4 outputs left out)

BERTScore unavailable: RobertaProcessing.__new__() got an unexpected keyword argument 'cls'
steps/block=8 blocks=8 temp=0.0 rep=1.15 -> {'rougeL': 0.1758, 'bleu4': 0.0479, 'bertscore': None}


BERTScore unavailable: RobertaProcessing.__new__() got an unexpected keyword argument 'cls'
steps/block=8 blocks=8 temp=0.3 rep=1.15 -> {'rougeL': 0.2201, 'bleu4': 0.0921, 'bertscore': None}


best decoding: (8, 8, 0.3, 1.15)



SAMPLE
PROMPT TAIL: opinion clinches the matter. To that effect, the CAT's decision that the entry of departmental candidates to the cadre of Postal Assistant is by way of direct recruitment is unexceptionable. We consequently affirm the findings of the CAT in the 
GENERATED: impugned of the High Court." 14 We uphold the the the decision of the Tribunal'ble the High Court of Karnataka in W.P.(CAT) 432/2014 in its order dated 14.08.2015 in the aforesaid W.P. observed as follows: "8. In the light of all the above, the the Hon of the Hon'ble the High Court of Karnataka in the case of Union of India and Ors Vs. Shakeel Ahmad Burney has upheld the the said order of the Principal Bench of the Tribunal the order of the Principal Bench of the Tribunal and passed the the order of the Principal Bench of the the the the order of the Principal Bench of the the the order of the Principal Bench of the the order of the Principal Bench of the the 
REFERENCE: impugned order. 14. As already held in the

## 10. Test predictions and Codabench zip

In [10]:
test_path = hf_hub_download(HF_REPO, test_file, repo_type="dataset", token=HF_TOKEN, local_dir=TMP)
test_data = json.load(open(test_path, encoding="utf-8")); ids = [str(k) for k in test_data]
print("test examples:", len(ids))
preds = generate([test_data[k]["text"] for k in ids], *BEST_DEC)
assert len(preds) == len(ids) and all(p.strip() for p in preds)
prediction = {k: {"text": p} for k, p in zip(ids, preds)}
PRED_PATH, ZIP_PATH = f"{WORK}/prediction.json", f"{WORK}/submission.zip"
json.dump(prediction, open(PRED_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z: z.write(PRED_PATH, arcname="prediction.json")
wl = [len(p.split()) for p in preds]
print("words per prediction (min/mean/max):", min(wl), np.mean(wl), max(wl), "| predictions under 100 words:", sum(w < 100 for w in wl))
print("saved:", ZIP_PATH, f"| total notebook time {(time.time()-T0)/3600:.2f} h")

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

test examples: 1000


words per prediction (min/mean/max): 11 126.456 128 | predictions under 100 words: 21
saved: /kaggle/working/cs781_phase1_diffusion/submission.zip | total notebook time 5.67 h
